In [ ]:
import pandas as pd   
import matplotlib.pyplot as plt
import numpy as np
import gc 
import seaborn as sns
import warnings


 # Memory Review Notes

 
  ## 2. Cell 16

  **Variable or expression**
  `meter_frames = {"Electricity": train[train["meter"] == 0], ...}`

  **Why expensive**
  This creates four large filtered DataFrames from a source table with roughly 20 million rows and keeps them alive at the same time. That duplicates substantial parts of `train` in memory.

  **Safer alternative**
  Loop over meter codes and filter inside the loop, reusing a single temporary variable. A better option is to aggregate directly with something like
  `groupby(["meter", pd.Grouper(...)])` instead of materializing four separate frames.

  ---

  ## 3. Cell 27

  **Variable or expression**
  `steam_frame = train[train["meter"] == 2]`
  followed by
  `steam_frame = steam_frame.merge(building_metadata, ...)`

  **Why expensive**
  A large steam-only copy is created first, then `merge()` allocates another DataFrame for the joined result. In notebook memory, both the filtered frame
  and merged frame may coexist longer than intended.

  **Safer alternative**
  Select only the required columns before the merge, reuse the same variable carefully, and delete intermediates after use with `del steam_frame`. If
  later steps no longer need the object, call `gc.collect()` as well.

  ---

  ## 4. Cell 24

  **Variable or expression**
  Repeated filters inside `calculate_zero_reading_percentage()`:

  - `train[train["meter"] == meter_type]`
  - `train[(train["meter"] == meter_type) & (train["meter_reading"] == 0)]`

  **Why expensive**
  Each call scans the full 20M-row DataFrame multiple times and creates temporary boolean masks and filtered views or copies. This is more CPU-heavy than
  catastrophic memory-heavy, but it is still inefficient in a notebook.

  **Safer alternative**
  Compute totals and zero counts once with grouped aggregations. For example, group by `meter` and use a shared zero-indicator mask or column instead of
  filtering repeatedly.

  ---

  ## 5. Cell 8

  **Variable or expression**
  `train[["timestamp", "meter_reading"]].set_index("timestamp").resample(...)` repeated twice

  **Why expensive**
  Two separate temporary two-column DataFrames are created, re-indexed, and resampled, then discarded. That repeats the same allocation and indexing work
  unnecessarily.

  **Safer alternative**
  Build the time-indexed subset once, reuse it for both resampling operations, or compute both aggregates from the same resample pipeline if possible.

  If you want, I can also turn this into a file like MEMORY_REVIEW.md in the repo.

In [ ]:
train = pd.read_csv("../data/train.csv", parse_dates=["timestamp"])
building_metadata     = pd.read_csv("../data/building_metadata.csv")
weather_data      = pd.read_csv("../data/weather_train.csv",     parse_dates=["timestamp"])

print("Train shape      :", train.shape)
print("Building shape   :", building_metadata.shape)
print("Weather shape    :", weather_data.shape)
print()
print(train.head(5))

In [ ]:
train.info()

train.isnull().sum()

In [ ]:
building_metadata.info()

## Distribution of Meter Types

In [ ]:
meter_labels = {
    0: "Electricity (0)",
    1: "Chilled Water (1)",
    2: "Steam (2)",
    3: "Hot Water (3)",
}

meter_counts = train["meter"].value_counts().sort_index()
labels = [meter_labels.get(meter, f"Meter {meter}") for meter in meter_counts.index]

plt.figure(figsize=(8, 8))
plt.pie(
    meter_counts,
    labels=labels,
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": "white"}
 )
plt.title("Meter Type Distribution")
plt.show()

As we can see above, most of the meter_reading is recorded as **0**. It holds  **90 percent** of all recordıng

#### METER READING DISTRIBUTION THROUGH TIMELINE 

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 6), dpi=100)
train[['timestamp', 'meter_reading']].set_index('timestamp').resample('h').mean()['meter_reading'].plot(ax=axes, label='By hour', alpha=0.8).set_ylabel('Meter reading', fontsize=14);
train[['timestamp', 'meter_reading']].set_index('timestamp').resample('D').mean()['meter_reading'].plot(ax=axes, label='By day', alpha=1).set_ylabel('Meter reading', fontsize=14);
axes.set_title('Mean Meter reading by hour and day', fontsize=16);
axes.legend();

#### DISTRIBUTION OF **0** METER READING VALUE FOR EACH METER TYPE

In [ ]:
zero_mask = train["meter_reading"] == 0
total_record = len(train)

count_electricity = ((train["meter"] == 0) & zero_mask).sum()
count_chilled_water = ((train["meter"] == 1) & zero_mask).sum()
count_steam = ((train["meter"] == 2) & zero_mask).sum()
count_hot_water = ((train["meter"] == 3) & zero_mask).sum()

print("Number of Electricity readings that are 0:", count_electricity)
print("Number of Chilled Water readings that are 0:", count_chilled_water)
print("Number of Steam readings that are 0:", count_steam)
print("Number of Hot Water readings that are 0:", count_hot_water)
print("---------------------------------")
print("Total record : ", zero_mask.sum())

In [ ]:
print("hello world")

In [ ]:
zero_count_chart = pd.DataFrame({
    "meter_type": ["Electricity", "Chilled Water", "Steam", "Hot Water"],
    "zero_count": [
        count_electricity,
        count_chilled_water,
        count_steam,
        count_hot_water,
    ],
})

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=zero_count_chart, x="meter_type", y="zero_count", palette="Blues_d")
plt.title("Number of 0 Meter Readings by Meter Type")
plt.xlabel("Meter Type")
plt.ylabel("Zero Reading Count")
plt.ylim(0, zero_count_chart["zero_count"].max() * 1.12)

for index, value in enumerate(zero_count_chart["zero_count"]):
    ax.text(index, value + zero_count_chart["zero_count"].max() * 0.01, f"{value:,}", ha="center")

plt.show()

##### Bu değerler normal veri setimize eklenecek  (merge ıslemı)
- building_metadata
- weather_data
 

## SOME QUESTIONS 
Why steam_meter type contains zero values at the one time interval ? 
Why chilled_water have noise between September and Novemeber ?

 ### Eda için yapılanlar 
- Tüm veri setindeki meter readingin zamana göre dağılımı 
-  0 meter readıng kaydedılen degerlerın meter type'a gore dagılımı
- Tüm gözlemlenen değerler için meter type dağılımı
- 0 meter reading kaydedilen değerlerin hafta sonu ve hafta içi dağılım tespiti
- Tüketilen toplam enerjinin günlere bağlı dağılımı
- Meter type'a bağlı olarak zaman içerisindeki ortalama enerji tüketimi 
- **Zamana** bağlı tüm meter typelara göre ortalama **zamana** bağlı tüm meter typelar ile benzer eğilim gösteriyor.



##### Eda için başka yapılabılecekler 
- site-id meter_reading değerinin dağılımı buıldıng_metadata merge edılecek. (zamana gore)
- endustrılerın toplam kayıtlarda dagılımı 
- (her endustrı ıcın readıng degerının dagılımı) 
- (her bır endustrının ortalama farkına bakılabılır)
- (her bır endustrının traın setınde kaydedılen zamana gore elektrık kullanımı test edılebılır)
- (her bır endustrının hafta ıcı veya hafta sonu gunlerıne gore elektrık kullanımı dagılımı bulunabılır)

- her bir meter_type'ın ortalama elektrık kullanımı (zamana gore)
- bina_id'sine göre ortalama elektrik kullanımı
- bina yaşına gore elektrık dagılımı testı yapılabılır. 

#### CHECK AVERAGE ENERGY CONSUMPTION FOR WEEKDAY (NOT ONLY 0 RECORDS)

In [ ]:
daily_total_consumption = (
    train.groupby(train["timestamp"].dt.normalize())["meter_reading"]
    .sum()
    .reset_index(name="gunluk_toplam_tuketim")
    .rename(columns={"timestamp": "gun"})
)

gun_isimleri = {
    0: "Pazartesi",
    1: "Sali",
    2: "Carsamba",
    3: "Persembe",
    4: "Cuma",
    5: "Cumartesi",
    6: "Pazar",
}

daily_total_consumption["haftanin_gunu_kodu"] = daily_total_consumption["gun"].dt.dayofweek
daily_total_consumption["haftanin_gunu"] = daily_total_consumption["haftanin_gunu_kodu"].map(gun_isimleri)

weekday_average_consumption = (
    daily_total_consumption.groupby(["haftanin_gunu_kodu", "haftanin_gunu"])["gunluk_toplam_tuketim"]
    .mean()
    .reset_index(name="ortalama_toplam_tuketim")
    .sort_values("haftanin_gunu_kodu")
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=weekday_average_consumption,
    x="haftanin_gunu",
    y="ortalama_toplam_tuketim",
    palette="Blues_d"
 )
plt.title("DAILY CONSUMPTION BY WEEKDAY")
plt.xlabel("WEEKDAY")
plt.ylabel("AVERAGE TOTAL CONSUMPTION")
plt.xticks(rotation=20)

plt.tight_layout()
plt.show()

#### CHECK METER READING VALUE FOR EACH METER TYPE

In [ ]:
meter_frames = {
    "Electricity": train[train["meter"] == 0],
    "Chilled Water": train[train["meter"] == 1],
    "Steam": train[train["meter"] == 2],
    "Hot Water": train[train["meter"] == 3],
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8), dpi=100)

for ax, (title, frame) in zip(axes.flat, meter_frames.items()):
    hourly = frame[["timestamp", "meter_reading"]].set_index("timestamp").resample("h").mean()["meter_reading"]
    daily = frame[["timestamp", "meter_reading"]].set_index("timestamp").resample("D").mean()["meter_reading"]
    hourly.plot(ax=ax, label="By hour", alpha=0.7)
    daily.plot(ax=ax, label="By day", alpha=0.9)
    ax.set_title(title)
    ax.set_ylabel("Meter reading")
    ax.legend()

plt.tight_layout()
plt.show()

#### IMPORTANT INSIGHT => STEAM METER READING SHOWS SAME PATTERN WITH ALL METER READING

In [ ]:
meter_avg = train.groupby("meter")["meter_reading"].mean()
meter_avg.index = ["Electricity", "Chilled Water", "Steam", "Hot Water"]

ax = meter_avg.plot(kind="bar", color="steelblue", figsize=(8, 4))
ax.set_title("Average Meter Reading by Meter Type")
ax.set_xlabel("Meter Type")
ax.set_ylabel("Average Reading")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
meter_avg

- LET'S CALCULATE KW CONSUMPTION FOR EACH METER TYPE    

In [ ]:
count_of_electric = train[train["meter"] == 0].shape[0]
count_of_chilled_water = train[train["meter"] == 1].shape[0]
count_of_steam = train[train["meter"] == 2].shape[0]
count_of_hot_water = train[train["meter"] == 3].shape[0]


print("Number of observations for Electricity:", count_of_electric)
print("Number of observations for Chilled Water:", count_of_chilled_water)
print("Number of observations for Steam:", count_of_steam)
print("Number of observations for Hot Water:", count_of_hot_water)

In [ ]:
def kw_to_observe_index(number_of_observation, mean_value) -> float:
    return mean_value / number_of_observation

meter_avg.shape[0]
count_list = [count_of_electric, count_of_chilled_water, count_of_steam, count_of_hot_water]
kw_index = {}

for i in range(meter_avg.shape[0]):
    index_value = kw_to_observe_index(count_list[i], meter_avg.iloc[i])
    kw_index[meter_avg.index[i]] = index_value
    print(f"{meter_avg.index[i]} - kW to Observe Index: {index_value:.6f}")
    




In [ ]:
kw_index_series = pd.Series(kw_index)

ax = kw_index_series.plot(kind="bar", color="teal", figsize=(8, 4))
ax.set_title("kW to Observe Index by Meter Type")
ax.set_xlabel("Meter Type")
ax.set_ylabel("kW to Observe Index")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
def calculate_zero_reading_percentage(meter_type) -> float:
    total_count = train[train["meter"] == meter_type].shape[0]
    zero_count = train[(train["meter"] == meter_type) & (train["meter_reading"] == 0)].shape[0]
    percentage = (zero_count / total_count) * 100 if total_count > 0 else 0
    return percentage

meter_types = {
    0: "Electricity",
    1: "Chilled Water",
    2: "Steam",
    3: "Hot Water" 
    }

for meter_code in meter_types:
    percentage = calculate_zero_reading_percentage(meter_code)
    print(f"Percentage of 0 readings for {meter_types[meter_code]}: {percentage:.2f}%")

##### kw_to_observe_index can be used as feature later in order to arrange understand ratio of meter_reading_mean_for_each_meterType / observe_count_for_each_type  .

##### SOME INFERENCES FOR THIS RESULT 
- Even if electricity has most observant, its kw_index is lowest 
- Steam caused energy consumption can be related to sector_id/primary_use/total_area 

In [ ]:
steam_frame = train[train["meter"] == 2]

steam_frame["building_id"]

## STEAM FRAME ICERISINDE OLAN VERILER HANGI SEKTOR AGIRLIKLI 

steam_frame = steam_frame.merge(building_metadata, on="building_id", how="left")

steam_frame.head()

